# Sentiment Analysis of US Airline Using LSTM

This approach combines deep learning (LSTM) to extract meaningful features to classify sentiments.

In [1]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
import gensim.downloader as api

In [2]:
# Load Dataset & membuang nilai null
df = pd.read_csv('/content/sample_data/Tweets.csv', usecols=['airline_sentiment', 'text']).dropna()

In [3]:
# Menyeimbangkan Jumlah Dataset (Negative, Neutral, Positive)
df_majority = df[df['airline_sentiment'] == 'negative']
df_minority = [df[df['airline_sentiment'] == label] for label in ['neutral', 'positive']]
df_balanced = pd.concat(
    [df_majority] + [resample(df_, replace=True, n_samples=len(df_majority), random_state=42) for df_ in df_minority]
).sample(frac=1, random_state=42).reset_index(drop=True)

In [4]:
# Encode Labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_balanced['airline_sentiment']).astype(np.int32)

In [7]:
# Optimized Text Preprocessing
clean_re = re.compile(r'http\S+|www\S+|[^a-zA-Z\s]')

def clean_text(text):
    return clean_re.sub('', text).lower().strip()

df_balanced['text'] = df_balanced['text'].astype(str).apply(clean_text)

In [8]:
# 🚀 Tokenization & Padding Optimization
OOV_TOK = "<OOV>"
max_length = 100

# Dynamically adjust vocab size
tokenizer = Tokenizer(oov_token=OOV_TOK)
tokenizer.fit_on_texts(df_balanced['text'])
vocab_size = len(tokenizer.word_index) + 1  # Use actual word count

X = pad_sequences(
    tokenizer.texts_to_sequences(df_balanced['text']),
    maxlen=max_length,
    padding='post',
    truncating='post',
    dtype=np.int32
)

In [9]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Feature Extraction Using Word2Vec (Glove)
# Load GloVe Embeddings
embedding_dim = 50
word_vectors = api.load("glove-twitter-50")
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size and word in word_vectors:
        embedding_matrix[i] = word_vectors[word]

[==================================================] 100.0% 199.5/199.5MB downloaded


In [11]:
# Optimized LSTM Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_length, trainable=True),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.2)),  # Reduce units for faster training
    Bidirectional(LSTM(32, dropout=0.2)),  # Fewer LSTM units, better speed
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
# Compilation & Training with 10 Epochs
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=5e-4), metrics=['accuracy'])
model.fit(X_train, y_train, validation_split=0.2, epochs=10, batch_size=16, verbose=2)  # Reduce epochs for faster convergence

Epoch 1/10
1102/1102 - 208s - 189ms/step - accuracy: 0.6678 - loss: 0.7602 - val_accuracy: 0.7581 - val_loss: 0.5878
Epoch 2/10
1102/1102 - 202s - 184ms/step - accuracy: 0.7819 - loss: 0.5613 - val_accuracy: 0.8000 - val_loss: 0.5099
Epoch 3/10
1102/1102 - 202s - 183ms/step - accuracy: 0.8225 - loss: 0.4652 - val_accuracy: 0.8243 - val_loss: 0.4585
Epoch 4/10
1102/1102 - 204s - 185ms/step - accuracy: 0.8513 - loss: 0.3978 - val_accuracy: 0.8461 - val_loss: 0.4089
Epoch 5/10
1102/1102 - 201s - 183ms/step - accuracy: 0.8756 - loss: 0.3422 - val_accuracy: 0.8600 - val_loss: 0.3889
Epoch 6/10
1102/1102 - 202s - 183ms/step - accuracy: 0.8938 - loss: 0.2976 - val_accuracy: 0.8677 - val_loss: 0.3720
Epoch 7/10
1102/1102 - 202s - 183ms/step - accuracy: 0.9072 - loss: 0.2641 - val_accuracy: 0.8770 - val_loss: 0.3569
Epoch 8/10
1102/1102 - 212s - 192ms/step - accuracy: 0.9188 - loss: 0.2335 - val_accuracy: 0.8833 - val_loss: 0.3540
Epoch 9/10
1102/1102 - 253s - 229ms/step - accuracy: 0.9316 - lo

In [13]:
# Evaluate Model
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", round(test_acc, 4))

# Save Model
model.save("lstm_sentiment_model.keras")

Test Accuracy: 0.896


In [14]:
# Optimized Prediction Function
def predict_sentiment(text, model, tokenizer, max_length=100):
    """
    Predict sentiment using LSTM model.
    """
    sequence = pad_sequences(
        tokenizer.texts_to_sequences([clean_text(text)]),
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    sentiment_label = np.argmax(model.predict_on_batch(sequence))  # Faster batch prediction
    return label_encoder.inverse_transform([sentiment_label])[0]

In [15]:
# Predict
loaded_model = tf.keras.models.load_model("lstm_sentiment_model.keras", compile=True)  # Ensure model is compiled
example_texts = [
    "I love flying with this airline! Their service is amazing.",
    "The flight was delayed for hours, very frustrating.",
    "It was okay, nothing special."
]

for text in example_texts:
    print(f"Text: {text} → Predicted Sentiment: {predict_sentiment(text, loaded_model, tokenizer)}")

Text: I love flying with this airline! Their service is amazing. → Predicted Sentiment: positive
Text: The flight was delayed for hours, very frustrating. → Predicted Sentiment: negative
Text: It was okay, nothing special. → Predicted Sentiment: negative
